# Quantization & Inference Optimization

## Why this matters

You experienced the *symptom* of this topic directly, in the very first version of the `local-rag-llm` project: local Ollama models (llama3.1, phi3) were taking 5-7 minutes per response on CPU, which is exactly the kind of problem quantization and inference optimization exist to solve. You fixed it by switching to the Claude API instead of fixing the local inference stack -- which was the right call for a personal project, but it means the *why it was slow* and *what the actual fixes would have been* never got covered. This notebook fills that gap: what was actually happening on your machine, and what the real levers are (most of which you now don't need to touch yourself, because you're on a hosted API -- but you should still know what's happening on the other side of that API call).

## Level 1: What it is

**Quantization**: reducing the numerical precision used to store a model's weights (and sometimes activations), trading a small amount of accuracy for large reductions in memory footprint and often speed. A model trained in 16-bit floating point (FP16/BF16) can be *quantized* down to 8-bit integers (INT8) or even 4-bit (INT4) after training, without retraining from scratch.

**Inference optimization** is the broader category quantization sits inside -- everything done to make a *trained* model respond faster or cheaper to run, without changing what it was trained to do. The other major levers, alongside quantization:

- **KV caching** -- reusing previously-computed attention key/value tensors instead of recomputing them for every new token
- **Batching** -- processing multiple requests' forward passes together on the same GPU
- **Speculative decoding** -- using a small, fast "draft" model to guess several tokens ahead, verified in bulk by the full model

## Level 2: How it actually works internally

### Why your local Ollama runs were 5-7 minutes on CPU

This is a genuinely useful concrete example to reason from. A model's weights are matrices of numbers, and generating each output token requires multiplying the current activations through every one of those weight matrices, layer by layer -- billions of floating-point multiply-accumulate operations per token, for a model in the multi-billion-parameter range. GPUs are built with thousands of cores designed specifically for this kind of massively parallel matrix math; a CPU has a handful of cores optimized for sequential, branchy logic instead. Running a 7-8B parameter model's matrix multiplications through a CPU means doing GPU-shaped work on the wrong hardware shape -- the *same total amount of arithmetic* takes vastly longer because far less of it happens in parallel per clock cycle. That's the root cause of the 5-7 minutes: not a bug, not bad configuration, just billions of operations running on hardware that processes them mostly serially.

### Quantization: how it reduces memory and speeds things up

A weight stored in FP16 takes 2 bytes; the same weight quantized to INT8 takes 1 byte, INT4 takes half a byte (packed two-per-byte). For a 7B parameter model, that's the difference between ~14GB (FP16) and ~3.5GB (INT4) just to hold the weights in memory. Two distinct benefits follow from this:

1. **Fits in less memory** -- a model that doesn't fit in available RAM/VRAM at FP16 might fit at INT4, which is often the actual blocker for running anything locally at all (this is likely part of what you hit -- a 7-8B model at FP16 needs more RAM than a typical laptop has to spare alongside everything else running).
2. **Faster arithmetic** -- integer operations are cheaper for hardware to execute than floating-point operations, and moving less data from memory to compute units (since each value is smaller) reduces a real bottleneck called memory bandwidth -- on many workloads, especially at low batch size, the GPU/CPU spends more time waiting for weights to arrive from memory than actually computing with them, so shrinking the weights speeds things up even beyond the raw arithmetic savings.

The trade-off: **quantization is lossy**. INT4 has a much coarser set of representable values than FP16, so the model's outputs shift slightly from the original -- usually a small, often imperceptible quality loss for INT8, more noticeable and task-dependent for INT4. Ollama's default quantization for most models it serves is around Q4 (a 4-bit variant) specifically to make consumer hardware viable -- which is a meaningful part of why local Ollama models can feel noticeably less sharp than the same-named model served at full precision via a hosted API.

### KV caching

Generating token N+1 requires attending back over all N previous tokens. Naively, that means recomputing the attention key and value vectors for every one of those N tokens, every single time you generate one more token -- wildly redundant, since tokens 1 through N-1 haven't changed. The KV cache stores those key/value vectors after they're computed once, so generating token N+1 only computes the new key/value pair for the newest token and reuses everything else from cache. This is why generation speed is roughly constant per token early in a response but can slow down as the context grows -- the cache itself grows linearly with sequence length, and reading a larger cache costs more, even though it avoids the much larger cost of recomputing from scratch.

### Batching

A GPU processing one request at a time is often *underutilized* -- the matrix operations for a single sequence don't fill all the parallel compute the hardware has. Batching packs multiple independent requests' computations together into one larger matrix operation, so the same GPU pass serves several users' tokens at once. This is purely a serving-infrastructure technique -- it doesn't change what any individual response looks like, only how efficiently the hardware is used across many concurrent requests. This is exactly why API providers can serve many users cost-effectively in a way a single local Ollama instance processing one request alone cannot -- you weren't benefiting from batching at all on your local runs, since there was only ever one request in flight.

### Speculative decoding

A small, fast "draft" model predicts several tokens ahead (say, 4-5 tokens) in a fraction of the time the full model would take. The full model then verifies all of those draft tokens in a single forward pass -- checking "would I have generated this same sequence" -- which is cheaper than generating them one at a time, because verification can happen in parallel across the draft tokens while sequential generation cannot. Accepted tokens are kept; the first rejected token (if any) is where the full model takes back over token-by-token generation from that point. Net effect: faster generation on the sequences where the draft model's guesses are often correct, with no quality loss, since the full model still verifies everything -- the speedup comes from parallelizing verification, not from trusting the smaller model's judgment.

## Level 3: Where it breaks, and what it doesn't solve

1. **Quantization quality loss is real and task-dependent, not a fixed universal cost.** Some tasks barely notice INT4 quantization (casual conversation, summarization); tasks needing precise reasoning, exact numeric output, or careful multi-step logic tend to degrade more noticeably. This matters directly for something like the X++ review agents: if you were ever to self-host a quantized open model instead of using the Claude API, you'd want to specifically re-test whether the Security agent's precision held up at INT4, not just assume "quantization only costs a little quality" uniformly across every task.

2. **KV cache memory grows with context length, and this is a real production constraint.** Long conversations or large document contexts (like an entire X++ class pasted into a prompt) mean a larger KV cache held in memory for the duration of that request. At scale, this is often the actual limiting factor on how many concurrent requests a server can handle -- not raw compute, but memory occupied by KV caches for requests currently in flight. This is part of why very long context windows are expensive to serve, independent of how many tokens you're actually paying for.

3. **Batching improves throughput, not any single request's latency -- and can even slightly hurt it.** If your request gets batched alongside others, you may wait fractionally longer than a hypothetical dedicated GPU would give you, because the hardware is now serving several requests' matrix operations together. Providers tune batch size to balance overall throughput against per-request latency; you don't control this from the API caller's side.

4. **Speculative decoding's speedup is variable, not fixed.** It only pays off when the draft model's guesses are frequently correct -- highly predictable text (boilerplate, common phrasing) speeds up a lot; novel or unusual phrasing sees less benefit, since more draft tokens get rejected and generation falls back to the slower one-token-at-a-time path more often.

5. **None of this is something you need to implement yourself when calling a hosted API.** This is worth being explicit about, since it changes what "knowing this" is actually for in your case: Anthropic's infrastructure handles quantization, KV caching, batching, and any speculative decoding decisions on their side of `client.messages.create()`. Your actual lever as an API caller for cost/latency is prompt caching (next notebook) and how much you ask the model to generate -- not any of the techniques in this notebook directly. Know this topic to reason about *why* API latency/cost behaves the way it does, and to be credible in an interview about infrastructure trade-offs, not because you'll be tuning quantization bits yourself on a hosted-API project.

In [ ]:
# Rough memory-footprint math for the quantization comparison above --
# illustrative, not measuring anything live (no local model loaded here).

def approx_weight_memory_gb(param_count_billions: float, bits_per_param: int) -> float:
    bytes_per_param = bits_per_param / 8
    total_bytes = param_count_billions * 1e9 * bytes_per_param
    return total_bytes / (1024 ** 3)

for label, bits in [("FP16", 16), ("INT8", 8), ("INT4", 4)]:
    gb = approx_weight_memory_gb(param_count_billions=7, bits_per_param=bits)
    print(f"7B model at {label}: ~{gb:.1f} GB just for weights")

# FP16: ~13.0 GB
# INT8: ~6.5 GB
# INT4: ~3.3 GB
# This is weights only -- actual runtime memory also needs room for the
# KV cache (grows with context length) and activation memory, so real
# usage is higher than this figure, especially for long-context requests.

## Interview-ready summary

**Q: "Why was local inference slow for you, and what would you have needed to fix it properly?"**

Weak answer: "My laptop wasn't powerful enough so I switched to an API."

Strong answer: "Running a 7-8B parameter model on CPU meant doing inherently parallel matrix arithmetic on hardware built for sequential execution -- billions of multiply-accumulate operations per token, mostly serialized instead of parallelized across thousands of GPU cores. The real fixes would have been quantizing the model to reduce both memory footprint and the amount of data moved per operation, or running on a GPU where the same arithmetic is naturally parallelized. Ollama was already serving a Q4-quantized version by default, so quantization alone wasn't going to close a 5-7-minute gap -- the fundamental issue was CPU-vs-GPU, not precision. Switching to the Claude API was the right call for a personal project because it moved that entire problem to infrastructure someone else has already solved at scale -- quantization, KV caching, batching, all handled server-side. My actual lever as an API caller is prompt design and caching, not inference-level optimization."